In [12]:
"""
# GIN + PSO — Best Reported Configuration

This notebook trains **only the GIN model** using the best PSO-selected hyperparameters reported so far. PSO has already selected the hyperparameters; Adam trains the neural-network weights. The notebook does not rerun PSO or train the other baselines.
"""

import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import re, glob, copy, time, random, warnings, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINConv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    confusion_matrix, classification_report, f1_score
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# Reproducibility
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except RuntimeError:
        pass

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Seed:", SEED)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Dataset and preprocessing settings
INPUT_ROOT = "/kaggle/input/"
ROWS_PER_FILE_CAP = 1_00_000
MAX_TOTAL_ROWS = 3_00_000
CLASS_THRESHOLD = 10_000
OTHERS_LABEL = "Others"
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.80, 0.10, 0.10

# Best reported GIN + PSO hyperparameters
K_NEIGHBORS = 3
HIDDEN_DIM = 256
DROPOUT = 0.43
LEARNING_RATE = 0.0028143775705957007
WEIGHT_DECAY = 5.9945168393624304e-05
EPOCHS = 200
# Stop if validation Macro-F1 does not improve for this many consecutive epochs.
PATIENCE = 30
# Ignore tiny Macro-F1 changes smaller than this threshold.
MIN_DELTA = 1e-4

print({
    "k_neighbors": K_NEIGHBORS,
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs": EPOCHS,
})


def discover_data_files(root=INPUT_ROOT):
    if not os.path.isdir(root):
        raise FileNotFoundError(
            f"Dataset directory not found: {root}. Attach the CICIDS2017 dataset in Kaggle and verify its path."
        )
    paths = sorted(
        glob.glob(os.path.join(root, "**", "*.parquet"), recursive=True)
        + glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
    )
    if not paths:
        raise FileNotFoundError(f"No CSV or Parquet files found under {root}")
    return paths

def read_dataset_file(path, cap, seed):
    suffix = Path(path).suffix.lower()
    if suffix == ".parquet":
        frame = pd.read_parquet(path)
    elif suffix == ".csv":
        frame = pd.read_csv(path, low_memory=False)
    else:
        return pd.DataFrame()
    if len(frame) > cap:
        frame = frame.sample(n=cap, random_state=seed)
    return frame.reset_index(drop=True)

data_files = discover_data_files()
frames = []
for i, path in enumerate(data_files, start=1):
    frame = read_dataset_file(path, ROWS_PER_FILE_CAP, SEED + i)
    if not frame.empty:
        frames.append(frame)
    print(f"[{i}/{len(data_files)}] {os.path.basename(path)}: {len(frame):,} rows")
if not frames:
    raise ValueError("No readable rows found.")
df_raw = pd.concat(frames, ignore_index=True, sort=False)
if len(df_raw) > MAX_TOTAL_ROWS:
    df_raw = df_raw.sample(n=MAX_TOTAL_ROWS, random_state=SEED).reset_index(drop=True)
print("Loaded shape:", df_raw.shape)

def find_label_column(df):
    exact = [c for c in df.columns if c.strip().lower() == "label"]
    if exact:
        return exact[0]
    candidates = [c for c in df.columns if "label" in c.strip().lower()]
    if candidates:
        return candidates[0]
    raise ValueError(f"Could not detect label column. Columns: {list(df.columns)}")

target_col = find_label_column(df_raw)
print("Target column:", target_col)
print(df_raw[target_col].value_counts().head(20))


# Clean data
n_before = len(df_raw)
df_clean = df_raw.drop_duplicates().copy()
numeric_cols_all = df_clean.select_dtypes(include=[np.number]).columns
df_clean[numeric_cols_all] = df_clean[numeric_cols_all].replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=[target_col])
df_clean = df_clean.dropna(axis=0, how="any").reset_index(drop=True)
print(f"Rows after cleaning: {len(df_clean):,} (removed {n_before-len(df_clean):,})")

# Exclude identifier-like fields and helper/target columns from features
identifier_pattern = re.compile(
    r"(^id$|_id$|^flow.?id$|\bip\b|ip_?src|ip_?dst|mac|timestamp|time_?stamp|^time$)",
    flags=re.IGNORECASE,
)
identifier_cols = [
    c for c in df_clean.columns
    if c != target_col and identifier_pattern.search(str(c))
]
feature_candidate_cols = [
    c for c in df_clean.columns
    if c != target_col and c not in identifier_cols
    and c not in {"_original_label", "_grouped_label", "_y_encoded"}
]
categorical_cols = [
    c for c in feature_candidate_cols
    if df_clean[c].dtype == object or str(df_clean[c].dtype).startswith("category")
]
numerical_cols = [c for c in feature_candidate_cols if c not in categorical_cols]
for c in categorical_cols:
    encoder = LabelEncoder()
    df_clean[c] = encoder.fit_transform(df_clean[c].astype(str))
feature_cols = numerical_cols + categorical_cols

# Preserve classes with >=1,200 cleaned rows; combine all smaller classes into Others
original_labels = df_clean[target_col].astype(str)
original_counts = original_labels.value_counts()
kept_labels = sorted(original_counts[original_counts >= CLASS_THRESHOLD].index.tolist())
df_clean["_grouped_label"] = original_labels.where(original_labels.isin(kept_labels), OTHERS_LABEL)
label_encoder = LabelEncoder()
df_clean["_y_encoded"] = label_encoder.fit_transform(df_clean["_grouped_label"].astype(str))
class_names = list(label_encoder.classes_)
n_classes = len(class_names)
print("Retained individual labels:", kept_labels)
print("Grouped classes:", class_names)
print("Grouped class counts:\n", df_clean["_grouped_label"].value_counts())

X_full = df_clean[feature_cols].to_numpy(dtype=np.float64)
y_full = df_clean["_y_encoded"].to_numpy(dtype=np.int64)
if len(np.unique(y_full)) < 2:
    raise ValueError("Need at least two target classes after grouping.")


# Stratified train/validation/test split (80/10/10), fixed seed
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_full, y_full, test_size=VAL_RATIO + TEST_RATIO,
    stratify=y_full, random_state=SEED
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=y_temp, random_state=SEED
)

# Fit scaler on training features only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)
print("Split sizes:", len(y_train), len(y_val), len(y_test))
print("Feature count:", X_train.shape[1])

# Inverse-frequency class weights from TRAIN labels only
train_counts = np.bincount(y_train, minlength=n_classes)
if np.any(train_counts == 0):
    raise ValueError("A class is absent from the training split; check class grouping/splitting.")
class_weights_array = compute_class_weight(
    class_weight="balanced", classes=np.arange(n_classes), y=y_train
).astype(np.float32)
class_weights_tensor = torch.tensor(class_weights_array, dtype=torch.float32)
print("Class weights:", dict(zip(class_names, class_weights_array.round(4))))


def build_knn_edge_index(X, k):
    n = X.shape[0]
    if n < 2:
        raise ValueError("Each split must contain at least two nodes.")
    k_eff = min(int(k), n - 1)
    neighbors = NearestNeighbors(n_neighbors=k_eff + 1, algorithm="auto").fit(X)
    _, indices = neighbors.kneighbors(X)
    src, dst = [], []
    for i in range(n):
        for j in indices[i]:
            if j == i:
                continue
            src.extend([i, int(j)])
            dst.extend([int(j), i])
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return torch.unique(edge_index, dim=1)

def make_split_graph(X, y, k):
    return Data(
        x=torch.tensor(X, dtype=torch.float32),
        edge_index=build_knn_edge_index(X, k),
        y=torch.tensor(y, dtype=torch.long),
    )

print(f"Building separate train/validation/test k-NN graphs with k={K_NEIGHBORS}...")
t0 = time.time()
data_train = make_split_graph(X_train, y_train, K_NEIGHBORS)
data_val = make_split_graph(X_val, y_val, K_NEIGHBORS)
data_test = make_split_graph(X_test, y_test, K_NEIGHBORS)
print(f"Graph construction completed in {time.time()-t0:.1f}s")
print("Edges:", data_train.edge_index.shape[1], data_val.edge_index.shape[1], data_test.edge_index.shape[1])


# GIN architecture used for the reported best result
class GINBaseline(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout):
        super().__init__()
        mlp1 = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )
        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x  # raw logits for CrossEntropyLoss


def train_gin(model, train_graph, val_graph):
    model = model.to(device)
    train_graph = train_graph.to(device)
    val_graph = val_graph.to(device)
    weights = class_weights_tensor.to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    # Early stopping monitors validation Macro-F1, the model-selection metric.
    # Always restore the checkpoint with the highest validation Macro-F1.
    best_val_macro_f1 = -1.0
    best_epoch = 0
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(train_graph.x, train_graph.edge_index)
        loss = criterion(logits, train_graph.y)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(val_graph.x, val_graph.edge_index)
            val_loss = criterion(val_logits, val_graph.y)
            val_pred = val_logits.argmax(dim=1)
            val_macro_f1 = f1_score(
                val_graph.y.cpu().numpy(), val_pred.cpu().numpy(),
                average="macro", zero_division=0
            )

        history["train_loss"].append(float(loss.item()))
        history["val_loss"].append(float(val_loss.item()))
        history["val_macro_f1"].append(float(val_macro_f1))
        improved = val_macro_f1 > best_val_macro_f1 + MIN_DELTA
        if improved:
            best_val_macro_f1 = float(val_macro_f1)
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0 or improved or epoch == EPOCHS:
            print(
                f"Epoch {epoch:03d}/{EPOCHS} | train_loss={loss.item():.4f} "
                f"val_loss={val_loss.item():.4f} val_macro_f1={val_macro_f1:.4f} "
                f"best={best_val_macro_f1:.4f} @ {best_epoch}"
            )
        if epochs_without_improvement >= PATIENCE:
            print(
                f"Early stopping at epoch {epoch}: validation Macro-F1 did not "
                f"improve by at least {MIN_DELTA} for {PATIENCE} consecutive epochs."
            )
            break

    model.load_state_dict(best_state)
    print(
        f"Restored best validation checkpoint: epoch={best_epoch}, "
        f"Macro-F1={best_val_macro_f1:.5f}"
    )
    return model, history, best_epoch, best_val_macro_f1


def evaluate_gin(model, test_graph):
    model.eval()
    test_graph = test_graph.to(device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        logits = model(test_graph.x, test_graph.edge_index)
        probabilities = F.softmax(logits, dim=1).cpu().numpy()
    if device.type == "cuda":
        torch.cuda.synchronize()
    inference_time = time.perf_counter() - t0

    y_true = test_graph.y.cpu().numpy()
    y_pred = probabilities.argmax(axis=1)
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    try:
        auc = roc_auc_score(
            y_true, probabilities, average="macro", multi_class="ovr",
            labels=list(range(len(class_names)))
        )
    except ValueError as exc:
        print("ROC-AUC unavailable:", exc)
        auc = float("nan")

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "roc_auc_macro_ovr": auc,
        "inference_time_sec": inference_time,
    }
    print("\nTEST METRICS")
    for name, value in metrics.items():
        print(f"{name}: {value:.5f}")
    print("\nCLASSIFICATION REPORT\n")
    print(classification_report(
        y_true, y_pred, labels=list(range(len(class_names))),
        target_names=[str(c) for c in class_names], zero_division=0
    ))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred, labels=list(range(len(class_names)))))
    return metrics


# Train only the PSO-selected GIN configuration.
# Early stopping and checkpoint selection use validation data only; test data is
# evaluated once after training and is never used to decide when to stop.
set_seed(SEED)
in_dim = X_train.shape[1]
out_dim = n_classes
model = GINBaseline(in_dim, HIDDEN_DIM, out_dim, DROPOUT)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

model, history, best_epoch, best_val_macro_f1 = train_gin(
    model, data_train, data_val
)
metrics = evaluate_gin(model, data_test)

# Save model and the essential metadata
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)
model_path = os.path.join(OUTPUT_DIR, "GIN_PSO_optimized.pt")
config_path = os.path.join(OUTPUT_DIR, "GIN_PSO_optimized_config.json")
results_path = os.path.join(OUTPUT_DIR, "GIN_PSO_optimized_metrics.json")

torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": in_dim,
    "hidden_dim": HIDDEN_DIM,
    "num_classes": out_dim,
    "dropout": DROPOUT,
    "class_names": class_names,
    "feature_columns": feature_cols,
    "scaler": scaler,
    "label_encoder": label_encoder,
}, model_path)

config = {
    "model": "GIN + PSO-selected hyperparameters",
    "seed": SEED,
    "k_neighbors": K_NEIGHBORS,
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs_max": EPOCHS,
    "best_epoch": best_epoch,
    "best_validation_macro_f1": best_val_macro_f1,
    "split": {"train": TRAIN_RATIO, "validation": VAL_RATIO, "test": TEST_RATIO},
    "class_threshold": CLASS_THRESHOLD,
    "class_names": class_names,
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
with open(results_path, "w") as f:
    json.dump({"metrics": {k: float(v) for k, v in metrics.items()}, "config": config}, f, indent=2)
print("Saved model:", model_path)
print("Saved config:", config_path)
print("Saved metrics:", results_path)



Device: cuda
Seed: 42
GPU: Tesla T4
{'k_neighbors': 3, 'hidden_dim': 256, 'dropout': 0.43, 'learning_rate': 0.0028143775705957007, 'weight_decay': 5.9945168393624304e-05, 'epochs': 200}
[1/3] test.csv: 100,000 rows
[2/3] train.csv: 100,000 rows
[3/3] validation.csv: 100,000 rows
Loaded shape: (300000, 47)
Target column: label
label
DDoS-ICMP_Flood            46540
DDoS-UDP_Flood             34778
DDoS-TCP_Flood             29017
DDoS-PSHACK_Flood          26182
DDoS-SYN_Flood             26165
DDoS-RSTFINFlood           25862
DDoS-SynonymousIP_Flood    22858
DoS-UDP_Flood              21176
DoS-TCP_Flood              17118
DoS-SYN_Flood              13116
BenignTraffic               7051
Mirai-greeth_flood          6393
Mirai-udpplain              5680
Mirai-greip_flood           4908
DDoS-ICMP_Fragmentation     3042
MITM-ArpSpoofing            2047
DDoS-UDP_Fragmentation      1835
DDoS-ACK_Fragmentation      1805
DNS_Spoofing                1174
Recon-HostDiscovery          874
Name: 